# Output distributions and scoring

When a model answers, it isn't choosing a sentence and then typing it. At each step it produces
a probability distribution over the next *token* (a token is a chunk of text, usually a short
word or word-piece; see
[how tokenizers work](https://huggingface.co/docs/transformers/tokenizer_summary)), some rule
picks one token from that distribution, and the model repeats the whole thing for the token
after that.

So a lot of what we call a model's "behaviour" is really two separate things: the shape of that
distribution, and the rule we use to read a token out of it. This lab pulls the two apart and
gives you code to look at each. The last part builds a small multiple-choice evaluator, so you
can see for yourself that the core of a benchmark score is scoring each candidate answer and
counting how often the best one is correct.

**Duration:** 90 min. **Prerequisites:** Day 0. **GPU:** optional; a 0.5B model runs on CPU
in a couple of minutes. Nothing here trains, so every cell after the download takes seconds.

In [1]:
# Installs the lab package on Colab; skipped when it's already importable (e.g. a local editable install).
try:
    import fast  # noqa: F401
except ImportError:
    %pip install -q git+https://github.com/sg-ai-safety-hub/FAST.git@main#subdirectory=src/packages/fast
    pass

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 72.1 MB/s eta 0:00:00


In [2]:
import torch

from fast.colab import setup
from fast.labs.day1_models import output_distributions as lab
from fast.testing import exercise

setup(require_gpu=False)
model, tokenizer = lab.load()

python 3.13.15  ·  colab=True  ·  seed=0
gpu    Tesla T4 (14.6 GB)


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

loaded qwen-0.5b on cuda:0


## Part 1: reading the distribution

A *forward pass* is one run of the model over the input. It ends in one vector of *logits* per
position: an unnormalised score for every token in the *vocabulary* (the model's fixed set of
possible tokens), so the vector has shape `(vocab_size,)`. These terms are in the
[transformers glossary](https://huggingface.co/docs/transformers/glossary). `lab.logits_for`
runs the pass and hands back the last position's vector, the model's opinion about what comes
next. None of it is a probability yet. Look at the shape and the range.

In [3]:
logits = lab.logits_for(model, tokenizer, "The capital of France is")
print(f"logits: shape {tuple(logits.shape)}  (one score per vocabulary token)")
print(f"{logits.shape[0]} tokens in the vocabulary; values run from {logits.min():.1f} to {logits.max():.1f}")

logits: shape (151936,)  (one score per vocabulary token)
151936 tokens in the vocabulary; values run from -14.3 to 17.1


### Exercise: logits to probabilities

[Softmax](https://en.wikipedia.org/wiki/Softmax_function) turns a vector of logits into a
probability distribution of the same shape: it exponentiates each score and divides by the
total, so the numbers come out positive and sum to 1. Temperature divides the logits first:
below 1 it sharpens the distribution toward the top token, above 1 it flattens it toward uniform.

One thing to watch. Real logits aren't centred on zero, so computing `exp()` on a raw logit can
overflow to infinity before you've covered the vocabulary. The usual fix is to subtract the
largest logit first, which doesn't change the result but keeps the numbers in range.
`torch.softmax` already does this, so if you build on it you get the stable version for free.

In [4]:
@exercise
def next_token_probs(logits: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
    """Turn a 1D tensor of logits into a probability distribution over the vocabulary.

    Divide by `temperature` (always > 0) before normalising. Return a tensor of the same
    shape that is non-negative, sums to 1, and stays finite for logits of any magnitude.
    """

    softmax_temp = torch.nn.functional.softmax(torch.div(logits, temperature), dim=-1)

    return softmax_temp


lab.check_next_token_probs(next_token_probs)

✅ next_token_probs: 6 checks passed


In [5]:
probs = next_token_probs(logits)
print(f"probabilities: shape {tuple(probs.shape)}, sum {float(probs.sum()):.3f}  (a distribution over the vocabulary)")
for temperature in (0.5, 1.0, 2.0):
    print(f"\nT = {temperature}")
    lab.top_tokens(tokenizer, next_token_probs(logits, temperature), k=10)

probabilities: shape (151936,), sum 1.000  (a distribution over the vocabulary)

T = 0.5
  71.55%  ' Paris'
  12.43%  ' ______'
   4.04%  ':\n'
   2.77%  ':\n\n'
   2.45%  ' __'
   1.91%  ' located'
   1.48%  ' ____'
   1.02%  ' the'
   0.55%  ' ('
   0.43%  ' ['

T = 1.0
  29.14%  ' Paris'
  12.15%  ' ______'
   6.92%  ':\n'
   5.74%  ':\n\n'
   5.39%  ' __'
   4.76%  ' located'
   4.20%  ' ____'
   3.48%  ' the'
   2.55%  ' ('
   2.25%  ' ['

T = 2.0
   2.36%  ' Paris'
   1.52%  ' ______'
   1.15%  ':\n'
   1.05%  ':\n\n'
   1.01%  ' __'
   0.95%  ' located'
   0.89%  ' ____'
   0.81%  ' the'
   0.70%  ' ('
   0.65%  ' ['


### For those who want to know more: what `next_token_probs` computes

**Where it sits.** A language model is a function from a sequence of tokens $x_1, \dots, x_t$ to a probability distribution over the next token $x_{t+1}$:

$$
p_\theta(x_{t+1} \mid x_1, \dots, x_t)
$$

One forward pass gives you this at *every* position at once. The output has shape `(batch, positions, vocab)`, and because attention is causal, the vector at position $t$ depends only on tokens $1..t$. For generation you only need the last position, which is what `lab.logits_for` returns. For this lab's model the vector has 151,936 entries, a few hundred more than the 151,665 tokens its tokenizer defines; the extras correspond to no text.

What the network emits there is *logits*: unnormalised scores. `next_token_probs` is the step that turns them into the distribution above. It is a pure function of the logits, with no model and no weights in it, so temperature is applied *after* the forward pass and changes nothing inside the model. That is why two deployments of the same checkpoint can behave differently, and why the setting rarely appears in a model card.

Two things build on it later in the lab. Reading a token out of the distribution (greedy, sampling at a temperature, top-p) is what the rest of Part 1 is about. Scoring a whole string (Part 2) multiplies these conditionals along the string, $p(y \mid x) = \prod_t p(y_t \mid x, y_{<t})$, which is why the scorer sums log-probabilities.

**Definition.** For logits $z \in \mathbb{R}^V$ and a temperature $T > 0$, the probability of token $i$ is

$$
p_i(T) \;=\; \frac{\exp(z_i / T)}{\sum_{j=1}^{V} \exp(z_j / T)}
$$

$T = 1$ is the ordinary softmax. Temperature scaling goes back to knowledge distillation ([Hinton et al., 2015](https://arxiv.org/abs/1503.02531)), which you meet again on Day 3.

**What temperature does.** The ratio of any two tokens is

$$
\frac{p_i(T)}{p_j(T)} \;=\; \exp\!\Big(\frac{z_i - z_j}{T}\Big)
$$

so the log-odds between two tokens is their logit gap divided by $T$. Below 1 the gaps widen and the top token pulls ahead; above 1 they shrink and the distribution flattens. In the limits, $T \to 0$ puts all the mass on the highest logit and $T \to \infty$ approaches uniform. The *order* of tokens never changes, since dividing by a positive number preserves it. Greedy decoding (always take the top token) is the $T \to 0$ limit, and no temperature is involved in it.

**Why subtracting the maximum is safe.** Shifting every logit by the same constant $c$ leaves softmax unchanged:

$$
\frac{\exp(z_i - c)}{\sum_j \exp(z_j - c)} \;=\; \frac{e^{-c}\exp(z_i)}{e^{-c}\sum_j \exp(z_j)}
$$

Choosing $c = \max_j z_j$ makes the largest exponent $e^0 = 1$, so nothing overflows. `torch.softmax` does this for you.

**Mistakes to watch for.**

1. **Dividing the probabilities by $T$ instead of the logits.** Writing `softmax(logits) / temperature` looks close but is not a temperature at all. The result sums to $1/T$, so it is not a distribution, and the ratios between tokens do not change (the $p_0/p_1$ ratio stays 2.718 at every $T$), so nothing is reshaped. It also passes quick checks: it equals the right answer at $T = 1$, and the top probability moves in the expected direction. For logits $[2, 1, 0, -1]$:

   | $T$ | correct: `softmax(logits / T)` | wrong: `softmax(logits) / T` | wrong sum |
   | --- | --- | --- | --- |
   | 0.5 | 0.865, 0.117, 0.016, 0.002 | 1.288, 0.474, 0.174, 0.064 | 2.000 |
   | 1 | 0.644, 0.237, 0.087, 0.032 | 0.644, 0.237, 0.087, 0.032 | 1.000 |
   | 2 | 0.455, 0.276, 0.167, 0.102 | 0.322, 0.118, 0.044, 0.016 | 0.500 |

   Even if you renormalise the wrong version so it sums to 1, you get the $T = 1$ distribution back exactly, so temperature would do nothing. Two quick sanity checks: `probs.sum()` at a temperature other than 1, and whether the ratio between two tokens changes with $T$.
2. **Writing the softmax by hand without subtracting the maximum.** `exp(z) / exp(z).sum()` overflows to `inf` for large logits and returns `nan`. Real logits are not centred on zero.
3. **Passing $T = 0$.** It divides by zero. If you want greedy behaviour, take the argmax directly.


### Exercise (optional): how sure is it?

Nothing later in the lab depends on this one, so skip it if you're short on time and come back
after Part 3. It's a useful number to have, not a prerequisite for the scorer.

"The model is confident" sounds vague, but you can put a number on it.
[Shannon entropy](https://en.wikipedia.org/wiki/Entropy_%28information_theory%29) measures how
spread out a distribution is, in bits. It's 0 when one token holds all the probability, and it
grows as the probability spreads across more tokens: *n* bits means the model is as undecided as
it would be choosing uniformly among 2ⁿ tokens.

That number is worth having because a peaked distribution and a flat one mean different things.
When the next token is all but forced (finishing "the capital of France is"), almost all the
probability sits on one token and entropy is near zero. When many continuations are about as
plausible as each other, the probability spreads out and entropy climbs. So entropy is a quick
read on whether the model is committed to one answer or spreading its bet.

In [6]:
@exercise
def entropy_bits(probs: torch.Tensor) -> float:
    """Shannon entropy of a probability distribution, in bits.

    Base 2, so the answer reads as "undecided between this many tokens". Tokens with zero
    probability contribute nothing, and must not turn the result into `nan`.
    """
    nonzero_probs = probs[probs != 0]
    entropy_2 = -(nonzero_probs * torch.log2(nonzero_probs)).sum()
    return float(entropy_2)

lab.check_entropy_bits(entropy_bits)

✅ entropy_bits: 7 checks passed


In [7]:
for prompt in (
    "The capital of France is",
    "The best programming language is",
    "My favourite colour is",
    "2 + 2 =",
):
    probs = next_token_probs(lab.logits_for(model, tokenizer, prompt))
    print(f"{entropy_bits(probs):5.2f} bits   {prompt!r}")

 4.50 bits   'The capital of France is'
 6.43 bits   'The best programming language is'
 5.19 bits   'My favourite colour is'
 0.72 bits   '2 + 2 ='


In [8]:
for prompt in (
    "2 + 2 =",
    "214 + 314 =",
    "39184 + 4818521 =",
    "391385812 * 48192941",
    "(391385811 * 19498185) / 301851="
):
  probs = next_token_probs(lab.logits_for(model, tokenizer, prompt))
  print(f"{entropy_bits(probs):5.2f} bits   {prompt!r}")


 0.72 bits   '2 + 2 ='
 2.06 bits   '214 + 314 ='
 2.63 bits   '39184 + 4818521 ='
 4.17 bits   '391385812 * 48192941'
 4.07 bits   '(391385811 * 19498185) / 301851='


<details>
<summary>For those who want to know more: Shannon entropy</summary>

**Surprisal.** An event of probability $p$ carries $-\log_2 p$ bits of surprisal: a certain event ($p = 1$) carries 0, a fair coin flip ($p = 1/2$) carries 1 bit, and rarer events carry more. The logarithm is there because surprisal should add for independent events: $-\log_2(pq) = -\log_2 p - \log_2 q$. It is the same reason Part 2 sums log-probabilities along a string.

**Entropy** is the *expected* surprisal when the outcome is drawn from the distribution itself:

$$
H(p) \;=\; -\sum_{i} p_i \log_2 p_i \;=\; \mathbb{E}_{i \sim p}\big[-\log_2 p_i\big]
$$

Three properties follow directly:

- $H(p) \ge 0$, with equality exactly when one token holds all the probability.
- Over $n$ tokens, $H(p) \le \log_2 n$, with equality exactly when the distribution is uniform.
- It depends only on the *shape* of the distribution: shuffling the tokens, or relabelling them, changes nothing.

**Reading a value in bits.** The quantity $2^{H}$ (the *perplexity*) is the number of equally likely tokens that would be exactly as undecided. A distribution $[0.7, 0.2, 0.1]$ has $H \approx 1.157$ bits and $2^{H} \approx 2.23$: it is about as undecided as a choice among 2.2 equally likely tokens, even though three tokens have probability. A uniform distribution over 8 tokens has exactly 3 bits, and a fair coin has 1 bit.

**Why zeros need care.** The term $0 \cdot \log 0$ is defined as 0, because $p \log p \to 0$ as $p \to 0$. In floating point, $\log_2 0 = -\infty$ and $0 \times (-\infty)$ is `nan`, so tokens with zero probability have to be skipped rather than passed through the formula.

**Bits and nats.** Bits use $\log_2$; nats use the natural logarithm. One nat is $1/\ln 2 \approx 1.4427$ bits, so bits = nats / $\ln 2$. Most training losses, and the scores from Part 2, are in nats, so convert before comparing them with a value in bits.

**Entropy and cross-entropy.** $H(p)$ is the expected surprisal when tokens really are drawn from $p$. When you instead score text that came from somewhere else with the model, as in Part 2, the average of $-\log q(\text{token})$ estimates the *cross-entropy* between the text and the model. This is what language-model training minimises, and it is never lower than the entropy of the source.

**Mistakes to watch for.**

1. **Natural log instead of $\log_2$.** The result is in nats, about 0.69 times the value in bits. The check recognises this.
2. **Letting zeros through.** `p * torch.log2(p)` with `p = 0` gives `nan`, which then poisons the whole sum.
3. **Dropping the minus sign.** Every term $p \log_2 p$ is $\le 0$, so without the minus the entropy comes out negative.
4. **Passing logits instead of probabilities.** Entropy needs a normalised distribution, so apply `next_token_probs` first.

</details>

### Exercise: top-p filtering

Top-p sampling, also called nucleus sampling, keeps the smallest set of the most likely tokens
whose probability adds up to `p`, and discards the rest before sampling
([Holtzman et al., 2019](https://arxiv.org/abs/1904.09751)). The reason it helps: the vocabulary
has tens of thousands of tokens, and although each unlikely token has little probability on its
own, together the long tail holds enough that you will sometimes draw from it. Those draws are
where off-topic or garbled tokens come from, so cutting the tail keeps generation coherent
without forcing the model to always take the single top token.

This matters when you read claims about how a model behaves. `p`, temperature and the other
sampling settings are chosen when the model is served, not fixed in the weights, and they change
the output. The same checkpoint run with different settings can behave noticeably differently,
and those settings usually aren't written in the model card.

In [9]:
@exercise
def top_p_filter(probs: torch.Tensor, p: float) -> torch.Tensor:
    """Zero out everything outside the nucleus and renormalise what's left.

    Keep the smallest set of highest-probability tokens whose mass reaches `p`. The token that
    takes the running total over the threshold is inside the nucleus, not outside it. Always
    keep at least one token, whatever `p` is.

    Return a tensor of the same shape as `probs`, summing to 1, with the surviving tokens in
    the same relative proportions they started in.
    """
    props_copy = torch.clone(probs)
    probs_sorted, probs_indexes = probs.sort(descending=True)
    cum_sum = len(probs_sorted[tuple([torch.cumsum(probs_sorted, dim=-1) < p])])
    total_p = cum_sum + 1 if cum_sum < len(probs_sorted) else cum_sum

    chosen = probs_indexes[:total_p]
    props_copy[:] = torch.tensor(tuple([0]))
    props_copy[tuple([chosen])] = probs[tuple([chosen])] / probs[tuple([chosen])].sum()

    return props_copy


lab.check_top_p_filter(top_p_filter)

✅ top_p_filter: 7 checks passed


In [10]:
probs = next_token_probs(logits)
for p in (0.9, 0.5):
    filtered = top_p_filter(probs, p)
    kept = int((filtered > 0).sum())
    print(f"p = {p}: filtered shape {tuple(filtered.shape)}, {kept} of {probs.shape[0]} survive, sum {float(filtered.sum()):.3f}")

p = 0.9: filtered shape (151936,), 25 of 151936 survive, sum 1.000
p = 0.5: filtered shape (151936,), 4 of 151936 survive, sum 1.000


In [ ]:
# Sampling is the usual reason a "deterministic" model gives different answers across runs; even
# at temperature 0, batch composition and non-deterministic GPU kernels can still slightly nudge the output.
nucleus = top_p_filter(next_token_probs(logits, temperature=1.0), 0.9)
draws = torch.multinomial(nucleus, num_samples=8, replacement=True)
print(f"nucleus: shape {tuple(nucleus.shape)}   draws: shape {tuple(draws.shape)}")
print("eight samples from the nucleus at T=1.0:")
print("  " + "  ".join(repr(tokenizer.decode([t])) for t in draws.tolist()))
print(f"greedy (argmax) always gives: {tokenizer.decode([int(probs.argmax())])!r}")

nucleus: shape (151936,)   draws: shape (8,)
eight samples from the nucleus at T=1.0:
  ' Paris'  ' __'  ' located'  ':\n'  ':\n\n'  ' Paris'  ' Paris'  ' located'
greedy (argmax) always gives: ' Paris'


*Greedy decoding* means taking the argmax (the single most likely token) at every step. It's
repeatable: the same prompt gives the same output every time. Any temperature above 0 adds
randomness to the choice, trading that repeatability for variety in what comes out. Hugging
Face's [generation strategies](https://huggingface.co/docs/transformers/generation_strategies)
walks through the common options.

## Part 2: scoring a string the model didn't write

So far we've sampled from the distribution to produce text. Scoring turns that around: instead
of asking the model to generate, you hand it a specific continuation and ask how much
probability it assigned to exactly that string. Nothing is generated.

This is more convenient than generating for most measurement tasks, because it gives you a
number you can compare directly instead of text you have to read and interpret. A multiple-choice
eval scores each option and takes the highest, which is what Part 3 does. Tomorrow's lab uses the
same scorer to measure which of two replies a model prefers. You write it once here and reuse it.

The mechanics: run the prompt and completion through together, and at each position read off the
[log probability](https://en.wikipedia.org/wiki/Log_probability) the model gave the token that
actually came next (the log of its softmax probability, so a negative number, and adding logs is
the same as multiplying probabilities). Sum those over the completion's tokens only. Position
*i*'s logits predict token *i+1*, so an off-by-one is the classic bug here. The check will catch
it.

The shapes the scorer moves through, before you write it:

In [43]:
p_ids = tokenizer("The capital of France is", return_tensors="pt").input_ids
c_ids = tokenizer(" Paris", add_special_tokens=False, return_tensors="pt").input_ids
both = torch.cat([p_ids, c_ids], dim=-1)
print(f"prompt ids       {tuple(p_ids.shape)}")
print(f"completion ids   {tuple(c_ids.shape)}")
print(f"concatenated     {tuple(both.shape)}   (batch, positions)")
with torch.no_grad():
    demo_logits = model(both.to(model.device)).logits
print(f"model logits     {tuple(demo_logits.shape)}   (batch, positions, vocab)")

prompt ids       (1, 5)
completion ids   (1, 1)
concatenated     (1, 6)   (batch, positions)


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [35]:
targets

tensor([[ 6722,   315,  9625,   374, 12095]])

In [34]:
prediction = torch.nn.functional.log_softmax(demo_logits, dim=-1)[:, :-1]
targets = both[:, 1:].to

In [42]:
demo_logits

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
@exercise
def sequence_logprob(model, tokenizer, prompt: str, completion: str) -> float:
    """Total log probability of `completion` following `prompt`, in nats (natural-log units).

    Sum over the completion's tokens only. The prompt is what you condition on, not something
    you score. One forward pass over the two concatenated is enough, with no generation.
    Return a plain `float`.
    """
    p_ids


lab.check_sequence_logprob(sequence_logprob, model, tokenizer)

In [ ]:
prompt = "Q: What is the capital of France?\nA:"
for answer in (" Paris", " Lyon", " Bangkok"):
    print(f"{sequence_logprob(model, tokenizer, prompt, answer):8.3f}   {answer!r}")

Usually what you want is the gap between two scores, since it tells you which continuation the
model finds more likely and by how much. One thing to watch before you rely on it. Log
probabilities are always negative, so every extra token pushes the total further down, and a
longer string ends up with a lower score just for being longer. Compare answers of different
lengths by their raw totals and the shortest one wins by default. Dividing each total by its
number of tokens, giving the average log probability per token, removes that bias. That's what
Part 3 does.

## Part 3: a multiple-choice evaluator from scratch

A multiple-choice benchmark is less involved than it sounds. You take a question and a few
candidate answers, score each answer as a continuation of the question with the function you
just wrote, and treat the highest-scoring answer as the model's pick. Do that over a set of
questions whose correct answers you already know, count how often the model's pick is right, and
that fraction is the accuracy. The big public benchmarks are the same procedure with more items:
[MMLU](https://arxiv.org/abs/2009.03300) has thousands, and
[lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) is the standard tool
that runs them.

Score each answer with the per-token average from Part 2, not the raw total. The candidate
answers have different lengths, and as we just saw, a raw sum would quietly favour the shortest
one whether or not it's correct.

In [ ]:
@exercise
def score_choices(model, tokenizer, question: str, choices: list[str]) -> list[float]:
    """Score each choice as a continuation of `question`, returning one number per choice.

    Use the mean per-token log probability of each choice, so a longer answer isn't penalised
    for its length. The evaluator picks the choice with the highest score. Build this on your
    `sequence_logprob` from Part 2.
    """
    raise NotImplementedError


lab.check_score_choices(score_choices, model, tokenizer)

In [ ]:
items = lab.mc_items()
correct = 0
for item in items:
    scores = score_choices(model, tokenizer, item["q"], item["choices"])
    pick = max(range(len(scores)), key=lambda i: scores[i])
    correct += int(pick == item["answer"])
    mark = "OK " if pick == item["answer"] else "MISS"
    print(f"{mark}  {item['q'].splitlines()[0]:36}  picked {item['choices'][pick]!r}")
print(f"\naccuracy: {correct}/{len(items)}")

## What you built

You now have three ways to look at a model's output distribution. Softmax with temperature turns
logits into probabilities and sets how peaked they are. Entropy measures how spread out they are.
Scoring reads off the probability of a specific string. The Part 3 evaluator is the scorer
run in a loop and checked against known answers, which is the core of what a benchmark number
measures.

Two things worth keeping. First, when a system card reports something like 74% on a benchmark,
you know the operation behind that number and some of the ways it can be shaped: how the answer
choices were worded, whether length bias was corrected, which tokens were counted. Second, the
output you get depends on decoding settings (temperature, top-p, greedy versus sampling) that are
chosen at serving time and sit outside the weights. The next lab picks up that thread: once you
see that the settings and the prompt around a model change what it does, a good part of a model's
apparent "behaviour" turns out to belong to whoever assembles those, not to the weights alone.

In [ ]:
# Stuck on sequence_logprob? Print the shapes. logits[0] has one row per input position, and
# row i scores the token at position i+1. So the rows you want end at the last position, and
# the target tokens start one index later.
print("row i predicts token i+1")